In [9]:
import numpy as np
import pandas as pd

In [3]:
suspects = ('pitcher', 'quaterback')
escape_vehicle = ('camry', 'bmw', 'bentley')
prior_probs = (.3, .7)
pitcher_probs = (.2, .43, .37)
qb_probs = (.1, .67, .23)

In [69]:
def draw_prior(suspects, probs, size=1):
    return np.random.choice(suspects, size=size, p=probs)


def draw_model(vehicles, suspects, pitcher_probs, qb_probs, num_sims):
    vehicles = np.array(vehicles)
    suspects = np.array(suspects)

    is_pitcher = (suspects == "pitcher")

    probs = np.where(
        is_pitcher[:, None],
        pitcher_probs,
        qb_probs
    )

    cumulative_probs = probs.cumsum(axis=1)
    random_v = np.random.rand(num_sims, 1)
    vehicle_index = (cumulative_probs > random_v).argmax(axis=1)

    return vehicles[vehicle_index]


def draw_joint(vehicles, suspects, prior_probs, pitcher_probs, qb_probs, size):
    s = draw_prior(suspects, prior_probs, size=size)
    w = draw_model(vehicles, s, pitcher_probs, qb_probs, num_sims=size)
    return s + "_" + w


def simulator(*args, num_sims=10000):
    return draw_joint(*args, size=num_sims)

In [71]:
num_sims = 100_000

sims = simulator(escape_vehicle, suspects, prior_probs, pitcher_probs, qb_probs, num_sims=num_sims)

scenarios, counts = np.unique(sims, return_counts=True)

In [74]:
for s, c in zip(scenarios, counts):
    print(f'{s} occurred with a probability of {round(c / num_sims, 2)}.')

pitcher_bentley occurred with a probability of 0.11.
pitcher_bmw occurred with a probability of 0.13.
pitcher_camry occurred with a probability of 0.06.
quaterback_bentley occurred with a probability of 0.16.
quaterback_bmw occurred with a probability of 0.47.
quaterback_camry occurred with a probability of 0.07.


In [73]:
# table to show approximate joint probabilities
probabilities = [round(c / num_sims, 2) for c in counts]

roles = []
cars = []

for s in scenarios:
    role, car = s.split("_")
    roles.append(role)
    cars.append(car)

df = pd.DataFrame({
    "Role": roles,
    "Car": cars,
    "Probability": probabilities
})

table = df.pivot(index="Role", columns="Car", values="Probability")
table

Car,bentley,bmw,camry
Role,,,
pitcher,0.11,0.13,0.06
quaterback,0.16,0.47,0.07
